# EDA: Raw Rainfall, Reanalysis, and DEM (Train/Val/Test)

This notebook performs exploratory data analysis (EDA) for:
- Raw rainfall targets
- Reanalysis patches
- DEM patches

It computes summaries both overall and across the deterministic time splits used by training:
- Train: 1980–2015
- Val:   2016–2020
- Test:  2021–2024


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd().resolve()
    p = start
    for _ in range(10):
        if (p / 'ML_Data_Preprocessing').exists() and (p / 'raw_data').exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError('Could not locate repo root containing ML_Data_Preprocessing/ and raw_data/')

repo_root = find_repo_root()
repo_root


In [ ]:
# Pick an assembled NPZ to analyze (prefer one-hot daily if present)
candidates = [
    repo_root / 'ML_Data_Preprocessing' / 'output' / 'assembled_npz' / 'full_training_data_daily_3x3_2km8km_one_hot.npz',
    repo_root / 'ML_Data_Preprocessing' / 'output' / 'assembled_npz' / 'full_training_data_daily_3x3_2km8km_cyclical.npz',
    repo_root / 'ML_Data_Preprocessing' / 'output' / 'assembled_npz' / 'full_training_data_monthly_3x3_2km8km_one_hot.npz',
    repo_root / 'ML_Data_Preprocessing' / 'output' / 'assembled_npz' / 'full_training_data_monthly_3x3_2km8km_cyclical.npz',
]

npz_path = next((p for p in candidates if p.exists()), None)
if npz_path is None:
    raise FileNotFoundError('No assembled NPZ found. Build via `python -m ML_Data_Preprocessing.assemble_training_data`')

npz_path


In [ ]:
z = np.load(npz_path, allow_pickle=True)
sorted(z.files)


In [ ]:
def arr(name: str):
    if name not in z.files:
        raise KeyError(f'Missing {name} in {npz_path.name}')
    return z[name]

stations = arr('stations').astype(object)
years = arr('years').astype(int)
months = arr('months').astype(int)
days = arr('days').astype(int) if 'days' in z.files else None
rain_mm = arr('rainfall_mm_raw').astype(np.float32)
re_patches = arr('reanalysis_patches').astype(np.float32)
variables = arr('variables').astype(object)

dem_local_raw = arr('dem_local_raw') if 'dem_local_raw' in z.files else np.array([])
dem_regional_raw = arr('dem_regional_raw') if 'dem_regional_raw' in z.files else np.array([])

if dem_local_raw.size == 0 or dem_regional_raw.size == 0:
    # Legacy fallback keys
    dem_local_raw = arr('dem_local_divstd') if 'dem_local_divstd' in z.files else arr('dem_local_minmax')
    dem_regional_raw = arr('dem_regional_divstd') if 'dem_regional_divstd' in z.files else arr('dem_regional_minmax')

N = int(len(rain_mm))
print('N samples:', N)
print('Reanalysis patches shape:', re_patches.shape, '(N,C,H,W)')
print('DEM local shape:', dem_local_raw.shape)
print('DEM regional shape:', dem_regional_raw.shape)
print('Variables (C):', len(variables))


## Split definition (same as training)

In [ ]:
idx_all = np.arange(N)
train_idx = idx_all[(years >= 1980) & (years <= 2015)]
val_idx = idx_all[(years >= 2016) & (years <= 2020)]
test_idx = idx_all[(years >= 2021) & (years <= 2024)]

splits = {
    'train': train_idx,
    'val': val_idx,
    'test': test_idx,
}
{k: len(v) for k, v in splits.items()}


In [ ]:
# Quick metadata sanity checks
print('Year range:', years.min(), years.max())
print('Month range:', months.min(), months.max())
if days is not None:
    print('Day range:', days.min(), days.max())

for name, idx in splits.items():
    st = pd.Series(stations[idx]).astype(str)
    print(f'{name}: n={len(idx)}, n_stations={st.nunique()}, years=[{years[idx].min()}..{years[idx].max()}]')


## Rainfall EDA (raw targets)

In [ ]:
def summarize_rain(x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=np.float32)
    return {
        'n': int(x.size),
        'mean': float(np.mean(x)),
        'std': float(np.std(x)),
        'min': float(np.min(x)),
        'p50': float(np.quantile(x, 0.50)),
        'p90': float(np.quantile(x, 0.90)),
        'p95': float(np.quantile(x, 0.95)),
        'p99': float(np.quantile(x, 0.99)),
        'max': float(np.max(x)),
        'pct_zero': float(np.mean(x == 0.0) * 100.0),
    }

pd.DataFrame({name: summarize_rain(rain_mm[idx]) for name, idx in splits.items()}).T


In [ ]:
# Histograms per split (raw mm)
bins = np.linspace(0, np.quantile(rain_mm, 0.995), 60)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, idx) in zip(axes, splits.items()):
    ax.hist(rain_mm[idx], bins=bins, alpha=0.8)
    ax.set_title(f'{name} rainfall (mm)')
    ax.set_xlabel('mm')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('count')
plt.tight_layout()


In [ ]:
# Log1p rainfall histogram (handles heavy tail)
bins = np.linspace(0, np.quantile(np.log1p(rain_mm), 0.995), 60)
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for ax, (name, idx) in zip(axes, splits.items()):
    ax.hist(np.log1p(rain_mm[idx]), bins=bins, alpha=0.8)
    ax.set_title(f'{name} log1p(rain)')
    ax.set_xlabel('log(1+mm)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('count')
plt.tight_layout()


In [ ]:
# Monthly seasonality (mean rain by month) across splits
fig, ax = plt.subplots(figsize=(10, 4))
for name, idx in splits.items():
    df = pd.DataFrame({'month': months[idx], 'rain': rain_mm[idx]})
    m = df.groupby('month')['rain'].mean().reindex(range(1, 13))
    ax.plot(m.index, m.values, marker='o', label=name)
ax.set_xticks(range(1, 13))
ax.set_title('Mean rainfall by month (mm)')
ax.set_xlabel('month')
ax.set_ylabel('mean mm')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()


## Station-level rainfall EDA

In [ ]:
# Per-station mean rainfall across splits
rows = []
for split, idx in splits.items():
    df = pd.DataFrame({
        'station': pd.Series(stations[idx]).astype(str),
        'rain': rain_mm[idx]
    })
    g = df.groupby('station')['rain'].agg(['count', 'mean', 'std'])
    g['split'] = split
    rows.append(g.reset_index())
station_stats = pd.concat(rows, ignore_index=True)
station_stats.sort_values(['split', 'mean'], ascending=[True, False]).head(10)


## Reanalysis EDA (patch statistics by variable)

In [ ]:
def patch_channel_mean(patches_nchw: np.ndarray) -> np.ndarray:
    # (N,C,H,W) -> (N,C)
    return np.nanmean(patches_nchw, axis=(2, 3))

re_mean = patch_channel_mean(re_patches)
re_mean.shape


In [ ]:
# Summary stats per variable by split
def summarize_var(x: np.ndarray) -> dict:
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {'mean': np.nan, 'std': np.nan, 'min': np.nan, 'p50': np.nan, 'p95': np.nan, 'max': np.nan}
    return {
        'mean': float(np.mean(x)),
        'std': float(np.std(x)),
        'min': float(np.min(x)),
        'p50': float(np.quantile(x, 0.5)),
        'p95': float(np.quantile(x, 0.95)),
        'max': float(np.max(x)),
    }

records = []
for split, idx in splits.items():
    m = re_mean[idx]  # (n,C)
    for ci, var in enumerate(variables.tolist()):
        stats = summarize_var(m[:, ci])
        stats.update({'split': split, 'var': str(var)})
        records.append(stats)
re_stats = pd.DataFrame(records)
re_stats.head()


In [ ]:
# Compare train vs test mean shift (per variable)
train_means = re_stats[re_stats['split'] == 'train'].set_index('var')['mean']
test_means = re_stats[re_stats['split'] == 'test'].set_index('var')['mean']
delta = (test_means - train_means).sort_values(key=lambda s: np.abs(s), ascending=False)
delta.to_frame('test_minus_train_mean').head(15)


In [ ]:
# Correlation (Pearson) between rainfall and each variable's patch-mean
def corr(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return float('nan')
    x = x[mask] - x[mask].mean()
    y = y[mask] - y[mask].mean()
    denom = float(np.sqrt(np.sum(x**2) * np.sum(y**2)))
    if denom == 0.0:
        return float('nan')
    return float(np.sum(x * y) / denom)

corr_rows = []
for split, idx in splits.items():
    y = rain_mm[idx]
    for ci, var in enumerate(variables.tolist()):
        x = re_mean[idx, ci]
        corr_rows.append({'split': split, 'var': str(var), 'corr': corr(x, y)})
corr_df = pd.DataFrame(corr_rows)
corr_df[corr_df['split'] == 'train'].sort_values('corr', key=lambda s: np.abs(s), ascending=False).head(10)


## DEM EDA

In [ ]:
# DEM patch means (local/regional)
dem_local_mean = np.nanmean(dem_local_raw.astype(np.float32), axis=(1, 2)) if dem_local_raw.ndim == 3 else np.nanmean(dem_local_raw.astype(np.float32), axis=1)
dem_regional_mean = np.nanmean(dem_regional_raw.astype(np.float32), axis=(1, 2)) if dem_regional_raw.ndim == 3 else np.nanmean(dem_regional_raw.astype(np.float32), axis=1)

def summarize(x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {'n': 0}
    return {
        'n': int(x.size),
        'mean': float(np.mean(x)),
        'std': float(np.std(x)),
        'min': float(np.min(x)),
        'p50': float(np.quantile(x, 0.5)),
        'p95': float(np.quantile(x, 0.95)),
        'max': float(np.max(x)),
    }

rows = []
for split, idx in splits.items():
    rows.append({'split': split, 'feature': 'dem_local_mean', **summarize(dem_local_mean[idx])})
    rows.append({'split': split, 'feature': 'dem_regional_mean', **summarize(dem_regional_mean[idx])})
pd.DataFrame(rows)


In [ ]:
# Correlation between rainfall and DEM means
for split, idx in splits.items():
    y = rain_mm[idx]
    c_local = corr(dem_local_mean[idx], y)
    c_reg = corr(dem_regional_mean[idx], y)
    print(split, 'corr(local_dem_mean, rain)=', c_local, 'corr(regional_dem_mean, rain)=', c_reg)


## Raw rainfall CSV coverage (optional)

In [ ]:
daily_dir = repo_root / 'raw_data' / 'daily_rainfall'
if daily_dir.exists():
    csvs = sorted(daily_dir.glob('*.csv'))
    print('daily_rainfall CSV files:', len(csvs))
    print('Example:', csvs[0].name if csvs else None)
else:
    print('No raw_data/daily_rainfall directory found (skipping)')


## Notes on your training curves

If train loss decreases but validation loss does not, that typically indicates one (or more) of:
- Distribution shift between train and val (especially in time-split data)
- Model overfitting quickly (capacity/regularization mismatch)
- Target is noisy / weakly predictable from available predictors

This notebook helps diagnose distribution shift by comparing rainfall, reanalysis, and DEM statistics across the splits.
